In [1]:
from  sentence_transformers import SentenceTransformer

In [2]:
model_name = "BAAI/bge-base-en-v1.5"

In [64]:
#model = SentenceTransformer(model_name)

In [3]:
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer

# model_name = "BAAI/bge-base-en-v1.5"
# model = SentenceTransformer(model_name)


# print(f"Device: {model.device}") 
# sentences = ["RAG is awesome", "Hello"]
# embeddings = model.encode(sentences)
# print(f"Embeddings shape: {embeddings.shape}")  


# from sklearn.metrics.pairwise import cosine_similarity
# similarity = cosine_similarity([embeddings[0]], [embeddings[1]])
# print(f"Similarity between sentences: {similarity[0][0]:.4f}")

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5", device='cpu')


print(model.device)

sentences = ["RAG is awesome", "Hello"]
embeddings = model.encode(sentences)
print(f"Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/777 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Cancellation requested; stopping current tasks.


KeyboardInterrupt: 

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.datasets import fetch_20newsgroups
import joblib 
import torch
import bm25s
import numpy as np
import pandas as pd

ModuleNotFoundError: No module named 'bm25s'

In [6]:
data_fetch=fetch_20newsgroups(subset='train' ,shuffle=True ,random_state=42 ,data_home='./dataset')

In [7]:
df = pd.DataFrame({
    'text' : data_fetch.data ,
    'category' : data_fetch.target
})

NameError: name 'pd' is not defined

In [5]:
data_fetch.target_names

['alt.atheism',
 'comp.graphics',
 'comp.os.ms-windows.misc',
 'comp.sys.ibm.pc.hardware',
 'comp.sys.mac.hardware',
 'comp.windows.x',
 'misc.forsale',
 'rec.autos',
 'rec.motorcycles',
 'rec.sport.baseball',
 'rec.sport.hockey',
 'sci.crypt',
 'sci.electronics',
 'sci.med',
 'sci.space',
 'soc.religion.christian',
 'talk.politics.guns',
 'talk.politics.mideast',
 'talk.politics.misc',
 'talk.religion.misc']

In [10]:
model_name="BAAI/bge-base-en-v1.5"

In [12]:

model =SentenceTransformer(model_name,device='cpu')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [24]:
embeddings =joblib.load('embeddings.joblib')

In [26]:
embeddings.shape

(11314, 768)

In [28]:
def cosine_similarity(v1 , array_of_vectors):

    if hasattr(v1,"detach"):
        v1=v1.detach().cpu().numpy()
    
    v1=np.asarray(v1,dtype=np.float32).ravel()

    if hasattr(array_of_vectors,"detach"):
        array_of_vectors=array_of_vectors.detach().cpu().numpy()
    
    A=np.asarray(array_of_vectors, dtype=np.float32)
    
    similarity_array=[]
    
    if A.ndim==1 :
        A.reval()
        denorm = np.linalg.norm(v1) * np.linalg.norm(A)
        return float(0.0 if denorm==0 else ((v1 @ A )/denorm) )
    A=np.atleast_2d(A)
    denorm=np.linalg.norm(v1) *  np.linalg.norm(A, axis=1)
    with np.errstate(divide='ignore' , invalid = 'ignore') :
        sims=A @ v1 / np.where(denorm==0 ,1.0 ,denorm)
    sims[denorm==0]=0.0
    return sims.tolist()


In [30]:
def euclidean_distance(v1 ,array_of_vectors) :
    v1=np.array(v1)
    if len(np.shape(array_of_vectors))==1 :
        array_of_vectors=[array_of_vectors]
    euclidean_distances=[]
    for v2 in array_of_vectors :
        v2=np.array(v2)
        dis=np.sqrt((np.sum(v1-v2)**2))
        euclidean_distances.append(float(dis))
    return euclidean_distances

In [40]:
def retrieve_relevant(que,embbeding,model,top_k=5): 
    que=clean_text(que)
    que_em=model.encode([que] ,convert_to_tensor=False).astype(np.float32)
    doc_em=embbeding
    for em in embbeding :
        if hasattr(em ,'detach'):
            em=em.detach().cpu().numpy()
        em=np.asarray(em ,dtype=np.float32)
    vals=[]
    
    simi= cosine_similarity(que_em,doc_em)
    top_k_result=top_k_relevant(simi,top_k)
    print(f"Query: {que}")
    for idx in top_k_result:
        print(f"Document: {df.iloc[idx]['text'][:2000]}...")
        print(f"Category: {data_fetch.target_names[df.iloc[idx]['category']]}...")
        print("\n\n")
    return vals

In [34]:
def top_k_relevant(lst,top_k) :
    lst=list(enumerate(lst))
    index_list=sorted(lst , key=lambda x : x[1] ,reverse=True )
    top_K_indices=[index for index ,value in index_list[:top_k]]
    return top_K_indices

In [36]:
def clean_text (text):
    text=text.strip()
    return text

In [62]:
example_query = "real madrid"

#result=retrieve_relevant(example_query, embeddings, model, top_k = 5)

In [52]:
def presicion_at_k(relevant_count ,k):
    if k<0 or relevant_count <0 :
        raise ValueError("all input must be non-nigative")
    if k==0 :
        return 0
    return  relevant_count/k

In [54]:
def recall_at_k(relevant_count , total_relevant_count ) :
    if relevant_count<0 or total_relevant_count <0 :
        raise ValueError ("all input must be non-nigative")
    if total_relevant_count==0 :
        return 0 
    return relevant_count/total_relevant_count

In [56]:
test_queries = [
    {"query": "advancements in space exploration technology", "desired_category": "sci.space"},
    {"query": "real-time rendering techniques in computer graphics", "desired_category": "comp.graphics"},
    {"query": "latest findings in cardiovascular medical research", "desired_category": "sci.med"},
    {"query": "NHL playoffs and team performance statistics", "desired_category": "rec.sport.hockey"},
    {"query": "impacts of cryptography in online security", "desired_category": "sci.crypt"},
    {"query": "the role of electronics in modern computing devices", "desired_category": "sci.electronics"},
    {"query": "motorcycles maintenance tips for enthusiasts", "desired_category": "rec.motorcycles"},
    {"query": "high-performance baseball tactics for championships", "desired_category": "rec.sport.baseball"},
    {"query": "historical influence of politics on society", "desired_category": "talk.politics.misc"},
    {"query": "latest technology trends in the Windows operating system", "desired_category": "comp.os.ms-windows.misc"}
    
]

In [100]:
def compute_metrics(queries ,embeddings ,model , top_k=5) :
    result=[]
    np_embeddings=[]
    for em in embeddings :  
        if hasattr(em,'detach'):
            em=embeddings.detach().cpu().numpy()
        np_embeddings.append(np.asarray(em,dtype=np.float32).ravel())
    E=np.vstack(np_embeddings)
    for item in queries :
        query =item["query"]
        desired_category=item["desired_category"]
        q_clean=clean_text(query)
        q_em=model.encode([q_clean] ,convert_to_tensor=False)
        q_em=np.asarray(q_em  , dtype=np.float32).ravel()
        
        simi_score=cosine_similarity(q_em,E)

        top_k_retrived=top_k_relevant(simi_score ,top_k)
        cat_in_retrived=[
            data_fetch.target_names[df.iloc[idx]["category"]] for idx in top_k_retrived
        ]

        relevant_in_top_k=sum(1 for cat in cat_in_retrived if cat==desired_category )

        total_relevant = sum(
            1 for idx in range(len(df))
            if data_fetch.target_names[df.iloc[idx]["category"]] == desired_category
        )
        presicion=presicion_at_k(relevant_in_top_k,top_k)
        recall=recall_at_k(relevant_in_top_k,total_relevant)
        result.append( {
            "query":query ,
            "precision@k" :presicion ,
            "recall@k" : recall
        })
    return result
        
        

In [104]:
k_values = [200, 20, 50]

for k in k_values:
    print(f"\n{'='*80}")
    print(f"Results with K={k}:")
    print('='*80)
    results = compute_metrics(test_queries, embeddings, model, top_k=k)
    
    for result in results:
        print(f"Query: {result['query']}")
        print(f"  Precision@{k}: {result['precision@k']:.2f}, Recall@{k}: {result['recall@k']:.2f}")
        print()


Results with K=200:
Query: advancements in space exploration technology
  Precision@200: 0.99, Recall@200: 0.33

Query: real-time rendering techniques in computer graphics
  Precision@200: 0.83, Recall@200: 0.28

Query: latest findings in cardiovascular medical research
  Precision@200: 0.91, Recall@200: 0.31

Query: NHL playoffs and team performance statistics
  Precision@200: 0.94, Recall@200: 0.31

Query: impacts of cryptography in online security
  Precision@200: 0.99, Recall@200: 0.33

Query: the role of electronics in modern computing devices
  Precision@200: 0.41, Recall@200: 0.14

Query: motorcycles maintenance tips for enthusiasts
  Precision@200: 0.92, Recall@200: 0.31

Query: high-performance baseball tactics for championships
  Precision@200: 1.00, Recall@200: 0.34

Query: historical influence of politics on society
  Precision@200: 0.39, Recall@200: 0.17

Query: latest technology trends in the Windows operating system
  Precision@200: 0.56, Recall@200: 0.19


Results with

In [11]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import Filter
from typing import List
from tqdm import tqdm
import joblib
import weaviate
import re
from weaviate.util import generate_uuid5
from pprint import pprint
import os

In [13]:
from utils import  suppress_subprocess_output

In [15]:
with suppress_subprocess_output() :
    client= weaviate.connect_to_embedded(
        persistence_data_path="./.collections" ,
        environment_variables={
            "ENABLE_API_BASED_MODULES": "true", 
            "ENABLE_MODULES": 'text2vec-transformers, reranker-transformers', 
            "TRANSFORMERS_INFERENCE_API":"http://127.0.0.1:5000/",
            "RERANKER_INFERENCE_API":"http://127.0.0.1:5000/"         }
    )

URLError: <urlopen error [Errno 110] Connection timed out>

In [ ]:
data = joblib.load("data.joblib")
print_object_properties(data[0])